# Fine-tuning LoRA/QLoRA sur A100 (Colab pay-as-you-go)

Notebook prêt à l'emploi pour enrichir un modèle récent (Gemma 4, Mistral...) avec tes propres données.

**Important avec des crédits pay-as-you-go (pas d'abonnement) :**
- Le compteur tourne dès que le runtime A100 est actif, même en pause. Pense à faire **Exécution > Se déconnecter et supprimer l'exécution** dès que tu as fini.
- Vérifie ton solde de crédits dans Colab avant de lancer un run long.
- Toutes les étapes lourdes (install, chargement modèle) ne sont à faire qu'une fois par session.

## 1. Vérifier le GPU attribué
Assure-toi d'avoir sélectionné A100 dans **Exécution > Modifier le type d'exécution > A100 GPU** avant de lancer cette cellule.

In [ ]:
!nvidia-smi

import torch
print("GPU dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_go = props.total_memory / (1024**3)
    print(f"GPU : {props.name} | VRAM totale : {vram_go:.1f} Go")
    USE_4BIT = vram_go < 60  # 4-bit conseillé si A100 40 Go, optionnel si 80 Go
    print("Quantization 4-bit recommandée :", USE_4BIT)
else:
    raise RuntimeError("Aucun GPU détecté — vérifie le type d'exécution.")

## 2. Installer les librairies

In [ ]:
!pip install -q unsloth trl peft transformers accelerate bitsandbytes datasets huggingface_hub

In [ ]:
pip install --upgrade transformers

In [ ]:
pip install --upgrade --no-cache-dir unsloth unsloth_zoo

## 3. Connecter Google Drive (pour sauvegarder ton travail)
Indispensable : si le runtime se coupe, tout ce qui n'est pas sur Drive/HF est perdu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

finetunning_path = 'finetuning'
project_name = 'cyber'

drive_path = f'/content/drive/MyDrive/{finetunning_path}/{project_name}'
os.makedirs(drive_path, exist_ok=True)
print(f"Dossier de travail prêt : {drive_path}")

## 4. Charger tes données

**Option A — déjà sur Drive** : dépose ton fichier `.jsonl` dans `/content/drive/MyDrive/finetuning/` puis passe à la cellule suivante.

**Option B — upload direct** : décommente et exécute la cellule ci-dessous.

Format attendu (une ligne JSON par exemple) :
```
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

In [ ]:
# Option B : upload direct (décommente si besoin)
# from google.colab import files
# uploaded = files.upload()
# import shutil
# for fname in uploaded.keys():
#     shutil.move(fname, f'{drive_path}/{fname}')

DATA_PATH = f"{drive_path}/dataset_for_autotrain_cyber.jsonl"  # <-- adapte le nom du fichier
assert os.path.exists(DATA_PATH), f"Fichier introuvable : {DATA_PATH}"
print("Fichier trouvé :", DATA_PATH)

## 5. (Optionnel) Connexion Hugging Face
Nécessaire si le modèle est gated (ex. certains repos Gemma) ou si tu veux push le résultat sur ton compte HF à la fin.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Charger le modèle de base

Change `MODEL_NAME` selon ce que tu veux fine-tuner : un modèle standard (`google/gemma-4-E4B-it`, `mistralai/Mistral-7B-Instruct-v0.3`...) ou une version abliterée trouvée sur le Hub (ex. `mlabonne/...-abliterated`). Vérifie le nom exact du repo sur huggingface.co avant de lancer.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Ensure USE_4BIT is defined
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_go = props.total_memory / (1024**3)
    USE_4BIT = vram_go < 60
else:
    USE_4BIT = False # Default or handle error if no GPU

MODEL_NAME = "OBLITERATUS/Gemma-4-12B-OBLITERATED"  # <-- change ici
MAX_SEQ_LENGTH = 8192                 # A100 permet de monter haut, ajuste selon tes données

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = USE_4BIT,   # défini automatiquement à l'étape 1
    dtype = None,              # auto (bf16 sur A100)
)

print("Modèle chargé :", MODEL_NAME)

## 7. Configurer le LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                     # rank : 16 (léger) à 32-64 (plus de capacité, A100 encaisse bien)
    target_modules = "all-linear",
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
)

model.print_trainable_parameters()

## 8. Préparer le dataset
Applique le chat template du modèle et vérifie un exemple avant de lancer l'entraînement — c'est l'erreur la plus fréquente.

In [ ]:
from datasets import load_dataset
import re # Ajouté pour l'analyse des chaînes

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print("Nombre d'exemples :", len(dataset))

def format_example(example):
    # Check if 'messages' key exists, which is the preferred format
    if "messages" in example:
        text = tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )
    # If not, try to reconstruct 'messages' from 'instruction' and 'output'/'response'
    elif "instruction" in example and ("output" in example or "response" in example):
        # Assume a simple instruction-response format
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example.get("output") or example.get("response")},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    # Handle the user's specific 'text' format: "### Instruction:\n...\n\n### Response:\n..."
    elif "text" in example:
        match = re.match(r"### Instruction:\n(.*?)\n\n### Response:\n(.*)", example["text"], re.DOTALL)
        if match:
            instruction = match.group(1).strip()
            response = match.group(2).strip()
            messages = [
                {"role": "user", "content": instruction},
                {"role": "assistant", "content": response},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        else:
            print(f"Warning: 'text' field does not match expected format. Example: {example['text'][:200]}. Keys found: {example.keys()}. Returning empty text.")
            return {"text": ""}
    else:
        # If neither format matches, print an informative error and the example keys
        print(f"Warning: Example missing 'messages', 'instruction'/'output', or 'text' keys. Keys found: {example.keys()}. Returning empty text.")
        # Return an empty text or handle as appropriate for your dataset
        return {"text": ""} # Return an empty text to avoid breaking the .map operation
    return {"text": text}

dataset = dataset.map(format_example)

print("\n--- Exemple formaté ---\n")
print(dataset[0]["text"][:1500])

## 9. Lancer l'entraînement

Ajuste `num_train_epochs` selon la taille du dataset :
- < 500 exemples → 1-2 epochs (risque de surapprentissage)
- 500-5000 → 2-3 epochs
- > 5000 → 1-2 epochs suffisent souvent

In [ ]:
import os
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = f"{drive_path}/outputs"

# --- Split train / eval (1% pour valider que le modèle généralise, pas juste mémorise) ---
dataset_split = dataset.train_test_split(test_size=0.01, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]
print(f"Train : {len(train_dataset)} exemples | Eval : {len(eval_dataset)} exemples")

# --- Détection d'un checkpoint existant pour reprise automatique ---
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
    ]
    if checkpoints:
        checkpoints.sort(key=lambda d: int(d.split("-")[-1]))
        last_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])

if last_checkpoint:
    print(f"✅ Checkpoint trouvé, reprise à : {last_checkpoint}")
else:
    print("ℹ️ Aucun checkpoint trouvé, démarrage d'un nouvel entraînement.")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 12,
        gradient_accumulation_steps = 2,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        bf16 = True,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 500,
        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 3,
        output_dir = OUTPUT_DIR,
        warmup_steps = 800,   # recalcule à ~5-10% du nombre total de steps si tu changes batch/epochs
        report_to = "none",
        optim = "adamw_8bit",
        # max_steps = 200,  # décommente pour un test rapide avant le run complet
    ),
)

trainer_stats = trainer.train(resume_from_checkpoint = last_checkpoint)

## 10. Test rapide du modèle fine-tuné

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [{"role": "user", "content": "Écris une phrase de test liée à ton domaine de données."}]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 11. Sauvegarder le résultat

**Fais au moins l'option A avant de couper le runtime.** Les autres sont selon ton usage cible.

In [ ]:
# --- Option A : sauvegarder l'adapter LoRA sur Drive (rapide, toujours faire ça) ---
LORA_PATH = f"{drive_path}/models/{project_name}_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print("Adapter LoRA sauvegardé :", LORA_PATH)

In [ ]:
# --- Option B : push direct sur ton compte Hugging Face (modèle mergé) ---
HF_REPO = f"nico248000000000/{project_name}"  # <-- adapte
model.push_to_hub_merged(HF_REPO, tokenizer, save_method="merged_16bit")
print("Poussé sur le Hub :", HF_REPO)

In [ ]:
# --- Option C : export GGUF pour usage local avec Ollama / llama.cpp ---
GGUF_PATH = f"{drive_path}/models/{project_name}_gguf"
model.save_pretrained_gguf(GGUF_PATH, tokenizer, quantization_method="q4_k_m")
print("Export GGUF terminé :", GGUF_PATH)

## 12. Ne pas oublier de couper le runtime

Avec des crédits pay-as-you-go, l'A100 consomme des crédits tant que le runtime tourne — même inactif.

**Exécution > Se déconnecter et supprimer l'exécution**, une fois tes fichiers confirmés sur Drive/HF.